# Preprocessing: Data Augmentation Pipeline

This notebook follows from `DW-EDA.ipynb` and produces the augmented dataset used by `Modeling.ipynb`.

The augmentation pipeline generates 5 augmented versions of each original volume:
- **Gamma Bright** (`--BRIGHT`): gamma=1.67
- **Gamma Dark** (`--DARK`): gamma=0.6
- **Noise** (`--NOISE`): Rayleigh noise injection (scale=0.67)
- **Low-Pass Filter** (`--LPF`): FFT-based 3D low-pass filter (radius=30)
- **Fan Distortion** (`--FAN`): Simulated OCT fan distortion (pivot_distance=121)

Result: 1110 original + 5×1110 = 6660 total volumes

## Configuration

In [ ]:
# Set to True to force regeneration of augmented files even if they already exist
FORCE_REGENERATE = False

# Paths
VOLUMES_CSV = '../p5_Modeling/volumes.csv'
OUTPUT_CSV = '../datasrc/volumeOCT-AUGMENTED_metadata.csv'

# Augmentation parameters
GAMMA_BRIGHT = 1.67
GAMMA_DARK = 0.6
NOISE_SCALE = 0.67
LPF_RADIUS = 30
FAN_PIVOT_DISTANCE = 121

## Imports

In [ ]:
import sys, os
import time
sys.path.append('..')
import numpy as np
import pandas as pd
import cv2
from octcv.mdl_lib import yX_split, XVolSet, describeArrayHTML
from IPython.display import display, HTML

## Load Dataset from DW-EDA

In [ ]:
df = pd.read_csv(VOLUMES_CSV)
print(f"Loaded {len(df)} volumes from {VOLUMES_CSV}")
display(df.head(3))

# Load volumes using display_volume column (uint8, not pre-normalized)
yl, y, X = yX_split(df, default_load_normalized=False)
X.describe()

## Augmentation Functions

In [ ]:
def append_filename(filepath, suffix=''):
    """Append a suffix to a filename before the extension."""
    filename = os.path.basename(filepath)
    base, ext = os.path.splitext(filename)
    dirpath = os.path.dirname(filepath)
    new_filename = base + suffix + ext
    return os.path.join(dirpath, new_filename)

def adjust_gamma(image, gamma=1.0):
    """Apply gamma correction to a volume (uint8).
    gamma > 1 --> brighter, gamma in (0,1) --> darker.
    """
    if image.ndim > 3:
        trushape = tuple([image.shape[dim] for dim in range(image.ndim) if image.shape[dim] > 1])
        image = image.reshape(trushape)
    invGamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    return cv2.LUT(image, table)

def injectRayleighNoise(volume, scale=0.67, result_range=(0, 255), match_total_intensity=True, preview=False):
    """Inject Rayleigh-distributed noise into a 3D volume."""
    noise = np.random.rayleigh(scale=scale * volume.std(), size=volume.shape)
    noisy = volume.astype(np.float64) + noise
    if match_total_intensity:
        noisy = noisy * (volume.sum() / noisy.sum())
    if result_range is not None:
        noisy = np.clip(noisy, result_range[0], result_range[1])
    return noisy.astype(np.uint8)

def low_pass_filter_3D(volume, mask_radius=30):
    """3D FFT-based low-pass filter (vectorized mask creation)."""
    f_transform = np.fft.fftn(volume)
    f_transform_shifted = np.fft.fftshift(f_transform)
    rows, cols, depth = volume.shape
    crow, ccol, cdepth = rows // 2, cols // 2, depth // 2
    z, y, x = np.ogrid[:rows, :cols, :depth]
    dist_from_center = np.sqrt((z - crow)**2 + (y - ccol)**2 + (x - cdepth)**2)
    low_pass_mask = (dist_from_center <= mask_radius).astype(float)
    filtered_f_transform = f_transform_shifted * low_pass_mask
    f_ishift = np.fft.ifftshift(filtered_f_transform)
    volume_filtered = np.fft.ifftn(f_ishift)
    volume_filtered = np.real(volume_filtered)
    return volume_filtered

def simulate_oct_fan_distortion(volume, pivot_distance=200):
    """Simulate OCT fan (beam divergence) distortion on a 3D volume.
    volume shape: (B, depth, width)
    """
    B, D, W = volume.shape
    cx = W / 2
    pivot_z = -pivot_distance
    x = np.arange(W)
    z = np.arange(D)
    xx, zz = np.meshgrid(x, z)
    dx = xx - cx
    dz = zz - pivot_z
    r = np.sqrt(dx**2 + dz**2)
    theta = dx / pivot_distance
    new_x = pivot_distance * np.sin(theta) + cx
    new_z = r * np.cos(theta) + pivot_z
    mapx = new_x.astype(np.float32)
    mapy = new_z.astype(np.float32)
    distorted = np.zeros_like(volume)
    for i in range(B):
        distorted[i] = cv2.remap(
            volume[i], mapx, mapy,
            interpolation=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_REFLECT
        )
    return distorted

## 1. Gamma Correction (Bright &amp; Dark)

In [ ]:
t0 = time.time()
brightPaths = []
darkPaths = []
skip_counter = 0
gen_counter = 0

print(f"Total of {len(X)} volumes to augment via gamma correction.\n")
for xvol in X:
    brightOutpath = append_filename(xvol.filepaths.iloc[0, 1], suffix='--BRIGHT')
    darkOutpath = append_filename(xvol.filepaths.iloc[0, 1], suffix='--DARK')
    
    if not FORCE_REGENERATE and os.path.isfile(brightOutpath) and os.path.isfile(darkOutpath):
        skip_counter += 1
        print(f"\r  {skip_counter} volumes already augmented (skipped).", end='')
    else:
        ogvol = xvol.load()[0, ..., 0]
        brightvol = adjust_gamma(ogvol, gamma=GAMMA_BRIGHT)
        darkvol = adjust_gamma(ogvol, gamma=GAMMA_DARK)
        np.save(brightOutpath, brightvol)
        np.save(darkOutpath, darkvol)
        gen_counter += 1
        print(f"\r  {gen_counter} gamma-augmented pairs saved.", end='')
    
    brightPaths.append(brightOutpath)
    darkPaths.append(darkOutpath)

elapsed = time.time() - t0
print(f"\n\nGamma augmentation complete. Generated: {gen_counter}, Skipped: {skip_counter}")
print(f"Time: {elapsed:.1f}s")

## 2. Noise Injection &amp; Low-Pass Filtering

In [ ]:
t0 = time.time()
noisyPaths = []
lpfPaths = []
skip_counter = 0
gen_counter = 0

print(f"Total of {len(X)} volumes to augment via noise/LPF.\n")
for xvol in X:
    noisyOutpath = append_filename(xvol.filepaths.iloc[0, 1], suffix='--NOISE')
    lpfOutpath = append_filename(xvol.filepaths.iloc[0, 1], suffix='--LPF')
    
    if not FORCE_REGENERATE and os.path.isfile(noisyOutpath) and os.path.isfile(lpfOutpath):
        skip_counter += 1
        print(f"\r  {skip_counter} volumes already augmented (skipped).", end='')
    else:
        ogvol = xvol.load()[0, ..., 0]
        nvol = injectRayleighNoise(ogvol, scale=NOISE_SCALE, result_range=(0, 255), match_total_intensity=True)
        lvol = low_pass_filter_3D(ogvol, LPF_RADIUS).astype(int)
        np.save(noisyOutpath, nvol)
        np.save(lpfOutpath, lvol)
        gen_counter += 1
        print(f"\r  {gen_counter} noise/LPF pairs saved.", end='')
    
    noisyPaths.append(noisyOutpath)
    lpfPaths.append(lpfOutpath)

elapsed = time.time() - t0
print(f"\n\nNoise/LPF augmentation complete. Generated: {gen_counter}, Skipped: {skip_counter}")
print(f"Time: {elapsed:.1f}s")

## 3. Fan Distortion

In [ ]:
t0 = time.time()
fanPaths = []
skip_counter = 0
gen_counter = 0

print(f"Total of {len(X)} volumes to augment via fan distortion.\n")
for xvol in X:
    fanOutpath = append_filename(xvol.filepaths.iloc[0, 1], suffix='--FAN')
    
    if not FORCE_REGENERATE and os.path.isfile(fanOutpath):
        skip_counter += 1
        print(f"\r  {skip_counter} volumes already augmented (skipped).", end='')
    else:
        ogvol = xvol.load()[0, ..., 0]
        fanvol = simulate_oct_fan_distortion(ogvol, pivot_distance=FAN_PIVOT_DISTANCE)
        np.save(fanOutpath, fanvol)
        gen_counter += 1
        print(f"\r  {gen_counter} fan-distorted volumes saved.", end='')
    
    fanPaths.append(fanOutpath)

elapsed = time.time() - t0
print(f"\n\nFan distortion augmentation complete. Generated: {gen_counter}, Skipped: {skip_counter}")
print(f"Time: {elapsed:.1f}s")

## 4. Assemble Augmented Metadata CSV

In [ ]:
# Build individual DataFrames for each augmentation type
# Each row maps an original volume's metadata to an augmented filepath

nonpath_cols = ['dx_class', 'glaucoma', 'PIN', 'laterality', 'left_eye', 'set']
base_meta = df[nonpath_cols].copy()

# Original
orig_df = base_meta.copy()
orig_df['augmentation'] = 'original'
orig_df['display_volume'] = df['display_volume'].values

# Bright
bright_df = base_meta.copy()
bright_df['augmentation'] = 'bright'
bright_df['display_volume'] = brightPaths

# Dark
dark_df = base_meta.copy()
dark_df['augmentation'] = 'dark'
dark_df['display_volume'] = darkPaths

# Noise
noise_df = base_meta.copy()
noise_df['augmentation'] = 'noise'
noise_df['display_volume'] = noisyPaths

# LPF
lpf_df = base_meta.copy()
lpf_df['augmentation'] = 'lpf'
lpf_df['display_volume'] = lpfPaths

# Fan
fan_df = base_meta.copy()
fan_df['augmentation'] = 'fan'
fan_df['display_volume'] = fanPaths

# Concatenate all
aug_df = pd.concat([orig_df, dark_df, bright_df, noise_df, lpf_df, fan_df], axis=0, ignore_index=True)

print(f"Augmented dataset assembled: {len(aug_df)} entries")
print(f"Augmentation types: {aug_df['augmentation'].unique()}")
print(f"\nClass distribution:")
print(aug_df.groupby(['augmentation', 'dx_class']).size().unstack(fill_value=0))

In [ ]:
aug_df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved augmented metadata to: {OUTPUT_CSV}")
print(f"  Columns: {list(aug_df.columns)}")
print(f"  Rows: {len(aug_df)}")

---

## Summary & Discussion

### What Was Done

This notebook implemented a **physics-informed data augmentation pipeline** to expand the original dataset of 1,110 OCT volumes to **6,660 volumes** (a 6× expansion). Five augmentation strategies were applied, each grounded in realistic sources of variation encountered in clinical OCT imaging:

| Augmentation | Rationale | Implementation |
|---|---|---|
| **Gamma Bright** (γ=1.67) | Simulates over-exposed scans or high signal-strength acquisitions | Power-law intensity transform |
| **Gamma Dark** (γ=0.60) | Simulates under-exposed scans or low signal-strength | Power-law intensity transform |
| **Rayleigh Noise** (scale=0.67) | Models speckle noise inherent to OCT interferometry | Additive Rayleigh-distributed noise |
| **Low-Pass Filter** (r=30) | Mimics defocus or reduced axial resolution | Fourier-domain circular mask |
| **Fan Distortion** (d=121) | Approximates geometric warping from scan-head misalignment | Radial pivot-based coordinate transform |

### Key Design Decisions

1. **Augmentation parameters were calibrated using EDA findings** from `DW-EDA.ipynb` — specifically the SSIM distributions and radial power spectra that revealed how much structural variation exists naturally across the dataset.
2. **Augmented volumes are saved to disk as `.npy` files** rather than generated on-the-fly during training. This trades disk space for reproducibility and faster data loading during model training.
3. **Class proportions are preserved** — each augmentation is applied uniformly across both classes (Normal and Glaucoma), maintaining the original 76%/24% glaucoma/normal ratio.
4. **A unified metadata CSV** (`augmented_metadata.csv`) tracks all 6,660 entries with augmentation type labels, enabling flexible subsetting during modeling.

### Interpretation

The augmentation budget of 6× was chosen as a conservative expansion that adds meaningful variation without risking the model memorizing augmentation artifacts. Each transform targets a different axis of real-world imaging variability:
- Gamma transforms cover **exposure variation**
- Rayleigh noise covers **sensor noise**
- LPF covers **resolution degradation**
- Fan distortion covers **geometric misalignment**

Together, these encourage the model to learn features that are robust to acquisition conditions rather than overfitting to the specific imaging setup of the original dataset.


### Conclusion

The preprocessing pipeline successfully generates a 6× expanded training set ready for consumption by `Modeling.ipynb`. The augmented volumes preserve the diagnostic content of the originals (verified via SSIM in EDA) while introducing controlled variation along clinically relevant axes. This positions the modeling phase to explore whether increased data volume—even when synthetically generated—can push model performance beyond what was achieved with the original 1,110 volumes in Part 1 of the project.

**Next step:** `Modeling.ipynb` — train and evaluate 3D CNN architectures on this augmented dataset.


In [ ]:
# Quick verification: check file existence for a sample
import random
sample_idx = random.randint(0, len(aug_df) - 1)
sample_path = aug_df.iloc[sample_idx]['display_volume']
exists = os.path.isfile(sample_path)
print(f"Sample verification:")
print(f"  Index: {sample_idx}")
print(f"  Path: {sample_path}")
print(f"  Exists: {exists}")
print(f"\nPreprocessing pipeline complete!")